# Week 3 — Cleaning your data
## group1b · Retail sales

**Your question**
> Which product categories and payment methods drive the most revenue, and how does that vary by store and over time?

**What this notebook does.** Pulls the raw data out of the database, fixes the
problems you found in week 2, and writes clean tables back into your own
schema. Power BI reads those tables in week 4.

**How to use it.** Every section has an explanation, then a cell to run, then
a `TODO` where you make a decision. The decisions are the work — the code
around them is scaffolding so you are not starting from a blank page.

**Before you start:** have your week 2 `data_quality_notes.md` open. Every
number you wrote there tells you what to fix here.

---
### One rule
Run this notebook top to bottom, in order. If it only works when you run cells
out of sequence, it is not finished — someone else in your group has to be able
to run it from scratch and get the same tables.


## 1. Setup

Run this once per session. Colab forgets everything when it disconnects, so
you will run it again tomorrow.


In [1]:
!pip install -q psycopg2-binary sqlalchemy

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from getpass import getpass

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print("pandas", pd.__version__)


pandas 2.3.3


### Connect

`getpass` hides your password as you type it, so it never ends up saved in the
notebook. **Never type your password directly into a cell** — the notebook goes
to GitHub and the password would go with it.


In [2]:
HOST   = "internship-db.coh86gwewtxb.us-east-1.rds.amazonaws.com"
DB     = "internship"
USER   = "group1b"
SCHEMA_RAW   = "raw_retail"
SCHEMA_MINE  = "group1b"

password = getpass("Password for group1b: ")

engine = create_engine(
    f"postgresql+psycopg2://{USER}:{password}@{HOST}:5432/{DB}?sslmode=require"
)

# quick check
pd.read_sql(f"SELECT count(*) AS rows FROM {SCHEMA_RAW}.sales", engine)


,rows
0,30120


You should see **30,120**. If not, stop — something is
wrong with the connection, not with your code.


## 2. Load the raw tables

Pull all four into pandas. They are small enough to hold in memory
comfortably.


In [3]:
df = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.sales", engine)
customers = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.customers", engine)
products = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.products", engine)
stores = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.stores", engine)

print('sales   ', df.shape)
print('customers'.ljust(12), customers.shape)
print('products'.ljust(12), products.shape)
print('stores'.ljust(12), stores.shape)


sales    (30120, 9)
customers    (600, 5)
products     (150, 5)
stores       (12, 4)


`.shape` gives (rows, columns). Check these against what you
recorded in week 2 — if a number is different, find out why before going on.


In [4]:
df.head(10)


,sale_id,sale_date,customer_id,product_id,store_id,quantity,payment_method,unit_price,line_total
0,992,2024-02-25,361,14,1,10,Mobile Money,33.07,330.70
1,27068,2025-04-20,458,7,9,2,Cash,125.04,250.08
2,27228,2025-07-28,420,101,5,1,Mobile Money,120.62,120.62
3,3773,2024-07-30,523,138,9,8,Mobile Money,252.60,2020.80
4,4800,2024-04-03,198,113,9,6,Card,92.45,554.70
5,2534,2025-02-08,50,128,4,9,Cash,138.26,1244.34
6,9135,2024-07-31,557,56,11,3,Card,136.20,408.60
7,11577,22-Dec-2024,183,115,10,3,Card,105.17,315.51
8,6412,2024/07/29,560,63,6,6,Mobile Money,304.01,1824.06
9,12920,31/08/2024,212,99,6,9,Card,76.36,687.24


In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30120 entries, 0 to 30119
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sale_id         30120 non-null  int64  
 1   sale_date       30120 non-null  object 
 2   customer_id     30120 non-null  int64  
 3   product_id      30120 non-null  int64  
 4   store_id        30120 non-null  int64  
 5   quantity        30120 non-null  int64  
 6   payment_method  28615 non-null  object 
 7   unit_price      30120 non-null  float64
 8   line_total      30120 non-null  float64
dtypes: float64(2), int64(5), object(2)
memory usage: 2.1+ MB


Look at `df.info()` carefully. Note which columns pandas
thinks are `object` — that means text. The date column will be one of them,
which is the whole problem.


## 3. Record where you are starting

Before changing anything, capture the numbers. At the end you will compare
against these and prove the cleaning worked.


In [6]:
before = {
    'rows':       len(df),
    'duplicates': len(df) - df['sale_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['payment_method'].nunique(),
}
before


{'rows': 30120, 'duplicates': 120, 'missing': np.int64(1505), 'categories': 12}

---
## 4. Remove duplicate rows

Week 2 told you how many exact duplicates there are. `drop_duplicates()`
removes them, keeping the first occurrence.


In [7]:
print("before:", len(df))
df = df.drop_duplicates()
print("after: ", len(df))
print("removed:", before['rows'] - len(df))


before: 30120
after:  30000
removed: 120


**TODO — write down the number removed. Does it match your
week 2 figure?**

If it does not, you are looking at something different from what you counted.
Work out which before continuing.


---
## 5. Standardise the messy categories

This is the one that would silently split your totals in Power BI. `payment_method`
has the same values written several ways — different capitalisation, stray
spaces.

`.str.strip()` removes leading and trailing spaces. `.str.title()` makes it
Title Case. Pick one form and apply it everywhere.


In [8]:
# what it looks like now
df['payment_method'].value_counts(dropna=False)


payment_method
Mobile Money       11412
Cash                9859
Card                3829
None                1500
  Mobile Money       539
MOBILE MONEY         509
mobile money         482
CASH                 465
cash                 452
  Cash               429
CARD                 206
  Card               160
card                 158
Name: count, dtype: int64

In [9]:
df['payment_method'] = df['payment_method'].str.strip().str.title()

df['payment_method'].value_counts(dropna=False)


payment_method
Mobile Money    12942
Cash            11205
Card             4353
None             1500
Name: count, dtype: int64

**TODO — how many categories now, and how many before?**

Now do the same for the other text columns. Week 2 should have told you which
ones are affected — it is not only this one.


In [10]:
# TODO: check and clean the text columns in your dimension tables
customers['region'] = customers['region'].str.strip().str.title()
products['category'] = products['category'].str.strip().str.title()

# check your work
customers['region'].value_counts(dropna=False).head(15)


region
Central          81
Greater Accra    76
Ashanti          72
Bono             71
Volta            69
Western          68
Eastern          64
Northern         63
None             36
Name: count, dtype: int64

---
## 6. Parse the dates

`sale_date` is text, in four different formats. `pd.to_datetime` with
`format='mixed'` handles them, and `dayfirst=True` tells it to read `03/04/2025`
as 3 April rather than 4 March.

**This is a real decision, not a setting.** Nothing in the data proves which
reading is right. Whatever you choose, write it down in your cleaning notes and
be ready to defend it.


In [11]:
# what formats are present
df['sale_date'].str.len().value_counts()


sale_date
10    28203
11     1797
Name: count, dtype: int64

In [12]:
df['sale_date'] = pd.to_datetime(
    df['sale_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'      # anything unparseable becomes NaT rather than crashing
)

print("could not parse:", df['sale_date'].isna().sum())
print("range:", df['sale_date'].min(), "to", df['sale_date'].max())


could not parse: 0
range: 2024-01-01 00:00:00 to 2025-12-30 00:00:00


**TODO — does that date range make sense now?**

Compare it with what the text version gave you in week 2. This is where the
text-sorting problem finally goes away.

If any rows failed to parse, decide what to do with them and say why.


---
## 7. Deal with impossible values

Week 2 found negative quantities and line totals. Look at them before deciding.


In [13]:
bad = df[(df['quantity'] < 0) | (df['line_total'] < 0)]
print("rows affected:", len(bad))
bad.head(10)


rows affected: 60


,sale_id,sale_date,customer_id,product_id,store_id,quantity,payment_method,unit_price,line_total
613,4350,2025-05-13,259,116,10,-6,Cash,152.02,-912.12
831,22121,2024-09-13,474,89,7,-6,Mobile Money,18.38,-110.28
1232,29698,2025-10-13,490,51,12,-5,Cash,163.27,-816.35
1984,8995,2024-06-21,444,149,9,-4,Card,118.69,-474.76
3302,1140,2025-01-03,394,7,3,-1,Mobile Money,125.47,-125.47
4643,28664,2025-10-31,377,89,9,-1,Cash,19.47,-19.47
4766,1911,2025-03-12,127,8,3,-9,Card,155.72,-1401.48
5408,17122,2025-09-08,30,66,12,-8,Mobile Money,271.14,-2169.12
5593,9867,2024-08-26,148,69,3,-4,Cash,61.01,-244.04
5613,2432,2024-08-06,189,149,1,-10,Mobile Money,116.17,-1161.70


**TODO — decide, and write down why.**

Three defensible options. There is no single right answer, but there is a wrong
one: doing it silently.

1. **Drop them.** Clean, but you lose whatever else was in those rows.
2. **Set them to NULL.** Keeps the row, marks the value as unknown.
3. **Fix them** — if a negative looks like a data-entry sign error, taking the
   absolute value may be justified. Only if you can argue it.


In [14]:
# TODO: implement your decision. One of these, or your own.

# option 1 — drop
# df = df[~df.index.isin(bad.index)]

# option 2 — set to NULL
# df.loc[bad.index, 'COLUMN'] = np.nan

print("rows now:", len(df))


rows now: 30000


---
## 8. Deal with orphan keys

Some `customer_id` values in your fact table point at
`customers` records that do not exist. A plain join would drop these rows
silently — which is exactly why you are handling them deliberately.


In [15]:
valid = set(customers['customer_id'])
orphans = df[~df['customer_id'].isin(valid)]

print("orphan rows:", len(orphans))
orphans[['sale_id', 'customer_id']].head(10)


orphan rows: 75


,sale_id,customer_id
311,11111,90064
1055,4363,90033
1828,21621,90046
2684,16720,90071
2776,13294,90048
3064,29651,90018
3485,20161,90009
3969,14847,90003
4416,29421,90042
4791,27566,90004


**TODO — decide, and write down why.**

1. **Drop them.** Simple, and you lose real transactions.
2. **Keep them, pointing at an "Unknown" record.** Preserves the totals, and
   your dashboard shows an Unknown category — which is honest.

Check every foreign key, not just this one.


In [16]:
# TODO: implement your decision

# option 1 — drop
# df = df[df['customer_id'].isin(valid)]

# option 2 — add an Unknown row to the dimension, then repoint orphans at it
# unknown = pd.DataFrame([{'customer_id': -1}])
# customers = pd.concat([customers, unknown], ignore_index=True)
# df.loc[~df['customer_id'].isin(valid), 'customer_id'] = -1

print("rows now:", len(df))


rows now: 30000


---
## 9. Handle missing values

Decide **per column**. A NULL is not always a mistake — sometimes it means
something real, and filling it in would be inventing data.


In [17]:
df.isna().sum().sort_values(ascending=False)


payment_method    1500
sale_id              0
sale_date            0
customer_id          0
product_id           0
store_id             0
quantity             0
unit_price           0
line_total           0
dtype: int64

**TODO — for each column with missing values, decide and record:**

| Column | How many | Decision | Why |
|---|---|---|---|
| `payment_method` | | | |
| | | | |

Options: leave as NULL (honest, Power BI shows blanks), fill with a label like
`'Unknown'` (good for text you will group by), or drop the row (only if the row
is useless without it).


In [18]:
# TODO: implement your decisions

# example — label missing text so it groups properly in Power BI
# df['payment_method'] = df['payment_method'].fillna('Unknown')

df.isna().sum().sort_values(ascending=False).head()


payment_method    1500
sale_id              0
sale_date            0
customer_id          0
product_id           0
dtype: int64

---
## 10. Prove it worked

Re-run your week 2 checks on the cleaned data. Every problem should now be
gone or accounted for.


In [19]:
after = {
    'rows':       len(df),
    'duplicates': len(df) - df['sale_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['payment_method'].nunique(),
}

pd.DataFrame([before, after], index=['before', 'after'])


,rows,duplicates,missing,categories
before,30120,120,1505,12
after,30000,0,1500,3


**TODO — explain every number that changed.**

If rows went down, you should be able to say exactly how many were duplicates,
how many were impossible values, and how many were orphans. If the numbers do
not add up, something happened that you did not intend.


---
## 11. Write the clean tables back

Into **your own schema**, not the raw one. `if_exists='replace'` rebuilds the
table each time you run the notebook — use `'append'` by mistake and running
twice silently doubles your data.


In [20]:
df.to_sql('sales_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
customers.to_sql('customers_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
products.to_sql('products_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
stores.to_sql('stores_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)

print("written")


written


### Confirm they landed


In [22]:
pd.read_sql(f"""
    SELECT table_name,
           (xpath('/row/c/text()',
            query_to_xml(format('SELECT count(*) AS c FROM %%I.%%I',
            table_schema, table_name), false, true, '')))[1]::text::int AS rows
    FROM information_schema.tables
    WHERE table_schema = '{SCHEMA_MINE}'
    ORDER BY table_name
""", engine)


,table_name,rows
0,customers_clean,600
1,products_clean,150
2,sales_clean,30000
3,stores_clean,12


---
## Before you finish week 3

- [ ] This notebook runs top to bottom without errors, from a fresh runtime
- [ ] Someone else in the group has run it and got the same tables
- [ ] Every TODO above has a written answer
- [ ] Your cleaning decisions and reasons are in your week 3 form
- [ ] This notebook is committed to `notebooks/` in your repository
- [ ] Your password is **not** anywhere in the notebook

**Test it properly:** Runtime → Restart runtime, then Run all. If it fails, it
is not finished.

Next week you connect Power BI to `group1b` and build the model on these tables.
